In [1]:
# Independent additional homework solution. Standard library, no model/API.
import json

RECORDS = {'REF-1001': {'owner': 'demo-user-1', 'status': 'under_review'},
           'REF-1002': {'owner': 'demo-user-2', 'status': 'approved'}}


def dispatch(call, trusted_user):
    # A supplied tool-call object is untrusted data, never executable code.
    if not isinstance(call, dict) or set(call) != {'name', 'arguments'}:
        return {'code': 'invalid_call'}
    if call['name'] != 'check_status':
        return {'code': 'tool_not_allowed'}
    args = call['arguments']
    if not isinstance(args, dict) or set(args) != {'reference'}:
        return {'code': 'invalid_arguments'}
    ref = args['reference']
    if not isinstance(ref, str) or len(ref) != 8 or not ref.startswith('REF-') or not ref[4:].isascii() or not ref[4:].isdigit():
        return {'code': 'invalid_arguments'}
    record = RECORDS.get(ref)
    # Same outward result for absent and unauthorized records.
    if record is None or record['owner'] != trusted_user:
        return {'code': 'unavailable'}
    return {'code': 'ok', 'status': record['status']}


def run_tool_homework():
    before = json.dumps(RECORDS, sort_keys=True)
    cases = [
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001'}}, {'code': 'ok', 'status': 'under_review'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1002'}}, {'code': 'unavailable'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-9999'}}, {'code': 'unavailable'}),
        ({'name': 'delete_record', 'arguments': {'reference': 'REF-1001'}}, {'code': 'tool_not_allowed'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001', 'owner': 'demo-user-2'}}, {'code': 'invalid_arguments'}),
        ({'name': 'check_status', 'arguments': {'reference': 1001}}, {'code': 'invalid_arguments'}),
        ('not a tool object', {'code': 'invalid_call'}),
    ]
    rows = []
    for index, (call, expected) in enumerate(cases, 1):
        actual = dispatch(call, trusted_user='demo-user-1')
        assert actual == expected, (index, actual, expected)
        rows.append({'case': index, 'result': actual, 'passed': actual == expected})
    assert json.dumps(RECORDS, sort_keys=True) == before
    result = {'mode': 'simulated_tool_protocol', 'passed': len(rows), 'total': len(cases),
              'records_unchanged': True, 'cases': rows}
    print(json.dumps(result, indent=2))
    return result




In [2]:
result = run_tool_homework()


{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 7,
      "result": {
        "code": "invalid_call"
      },
      "passed": true
    }
  ]
}


# Assignment A — Validate and Dispatch Structured Tool Calls

## 1. Objective

The objective of this assignment is to build a controlled dispatcher that validates structured tool calls before allowing a read-only application-status tool to execute.

The dispatcher accepts only the permitted tool `check_status`, validates its arguments, checks record ownership using a separately supplied trusted identity, and prevents unauthorized access.

## 2. Validation vs. Authorization

**Validation** checks whether the incoming tool call has the correct structure, required fields, and valid data types and values.

Examples:
- The call must be a dictionary containing exactly `name` and `arguments`.
- The tool name must be `check_status`.
- The arguments must contain exactly `reference`.
- The reference must be a string in the expected `REF-0000` format.

**Authorization** checks whether the trusted user has permission to access the requested record.

For example, `REF-1001` belongs to `demo-user-1`, while `REF-1002` belongs to `demo-user-2`. A valid reference does not automatically grant access to its record.

Therefore, validation checks whether a request is well-formed, while authorization checks whether the requester is allowed to access the resource.

## 3. Test Results

The dispatcher was tested against seven cases:

| Case | Scenario | Result |
|---|---|---|
| 1 | Owned reference | `ok` |
| 2 | Another user's reference | `unavailable` |
| 3 | Missing reference | `unavailable` |
| 4 | Disallowed tool | `tool_not_allowed` |
| 5 | Unexpected owner argument | `invalid_arguments` |
| 6 | Numeric reference | `invalid_arguments` |
| 7 | Invalid call object | `invalid_call` |

All seven test cases passed.

- Mode: `simulated_tool_protocol`
- Passed: `7`
- Total: `7`
- Records unchanged: `true`

## 4. Limitations

- The records are fictional and stored in a local Python dictionary.
- The demo identity is supplied separately for testing; it is not a real authentication system.
- This exercise uses manually supplied structured calls. It does not implement native function calling by a hosted model provider.
- The tool is read-only and does not modify external systems or real application records.

## 5. Conclusion

The dispatcher successfully validates structured calls, restricts tool access through an allowlist, checks ownership using a separate trusted identity, rejects invalid or unauthorized requests, and preserves the original records throughout all seven tests.